# Assignment 3 — Using Pinecone (Cloud Vector Database)

**Goal:** deploy a scalable, cloud-based vector database using Pinecone, and compare results to FAISS.

Unlike FAISS (Assignment 1) and Chroma (Assignment 2), which run **locally on this machine**, Pinecone is a **managed cloud service**: the vectors live on Pinecone's servers and every query travels over the network. We reuse the same `all-MiniLM-L6-v2` embeddings and the same "AI for environment" corpus so the results are directly comparable to FAISS.

In [1]:
%pip install pinecone sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import time
import numpy as np
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer

d:\nihal\datacamp\ML practice\ml_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1 — Connect to Pinecone

Paste your API key below. Pinecone api_key opens the connection to your cloud account.

In [ ]:
# PASTE YOUR KEY HERE (from the Pinecone dashboard -> API Keys)
API_KEY = "pcsk_7ABpGw_8GouYHLtyyeBwQ51Nv72AMTEVkm56fT8xVgEdu3UwEncAgQ5HEKqzTWpCnyyinf"

pc = Pinecone(api_key=API_KEY)
print("Connected to Pinecone")

Connected to Pinecone


## Step 2 — Build the corpus (50 documents)

The assignment requires **at least 50 documents**. We start from the 20 used in the FAISS task and add 30 more, all on the "AI for environment" theme. Each document also gets metadata (`source` and `year`) so we can try metadata filtering later.

In [ ]:
corpus = [
    "AI can predict flood risks by analyzing satellite imagery and weather patterns.",
    "Machine learning models detect illegal deforestation in the Amazon rainforest.",
    "Neural networks classify water quality from sensor data in real time.",
    "AI-powered drones monitor coral reef health and detect bleaching events.",
    "Deep learning models predict air pollution levels in urban areas.",
    "Natural language processing helps scientists extract climate data from research papers.",
    "Reinforcement learning optimizes energy consumption in smart grids.",
    "Computer vision detects plastic waste in oceans using satellite images.",
    "AI forecasting models help farmers optimize irrigation to reduce water usage.",
    "Machine learning detects methane leaks from oil pipelines using infrared imagery.",
    "AI models analyze weather patterns to improve renewable energy predictions.",
    "Deep learning classifies species in biodiversity surveys from camera trap images.",
    "AI tracks glacier retreat and ice cap changes using satellite analysis.",
    "Predictive models warn communities about wildfire spread using wind and moisture data.",
    "AI systems monitor soil health and recommend sustainable farming practices.",
    "Machine learning helps optimize recycling by automatically sorting waste materials.",
    "Computer vision detects oil spills in oceans from satellite imagery.",
    "AI models predict drought conditions weeks in advance using climate models.",
    "Deep learning identifies endangered species in wildlife photographs automatically.",
    "AI monitors deforestation rates by comparing historical and current forest cover.",
    "AI optimizes wind turbine placement to maximize clean energy generation.",
    "Machine learning forecasts solar power output from cloud cover data.",
    "Neural networks detect water leaks in city pipelines to prevent waste.",
    "AI analyzes ocean temperature data to study coral bleaching trends.",
    "Deep learning maps urban heat islands using thermal satellite images.",
    "AI predicts air quality index for cities to issue health warnings.",
    "Machine learning identifies polluted rivers from satellite color analysis.",
    "Computer vision counts wildlife populations from aerial drone footage.",
    "AI models simulate the impact of carbon emissions on global temperatures.",
    "Reinforcement learning controls building HVAC systems to cut energy use.",
    "AI detects illegal fishing vessels using satellite tracking and pattern analysis.",
    "Machine learning predicts crop yields under different climate scenarios.",
    "Deep learning monitors air pollution from traffic using street camera feeds.",
    "AI estimates forest carbon storage from LiDAR and satellite data.",
    "Computer vision spots early signs of plant disease to reduce pesticide use.",
    "AI forecasts flood-prone zones by combining rainfall and terrain data.",
    "Machine learning detects anomalies in power grids to integrate renewables.",
    "AI tracks plastic pollution flow in rivers using image recognition.",
    "Deep learning classifies cloud types to improve weather prediction.",
    "AI recommends optimal routes for electric buses to lower emissions.",
    "Machine learning models melting permafrost using temperature sensor networks.",
    "AI monitors endangered whale movements using underwater acoustic data.",
    "Computer vision inspects solar panels for damage using drone images.",
    "AI predicts landslide risk from rainfall and soil moisture readings.",
    "Deep learning detects bushfire smoke early from camera tower networks.",
    "AI optimizes water distribution in drought-affected regions.",
    "Machine learning estimates household energy waste from smart meter data.",
    "AI analyzes deforestation drivers using economic and satellite data together.",
    "Computer vision sorts compost from landfill waste at recycling plants.",
    "AI models sea level rise to help coastal cities plan defenses."
]

sources = ["scientific", "news", "research"]
metadatas = [
    {"title": f"doc{i}", "source": sources[i % 3], "year": 2018 + (i % 7)}
    for i in range(len(corpus))
]

print(f"Corpus size: {len(corpus)} documents")

Corpus size: 50 documents


## Step 3 — Generate embeddings

Same model as FAISS and Chroma: `all-MiniLM-L6-v2`, which turns each sentence into a **384-dimensional** vector. We will tell Pinecone to expect 384 dimensions when we create the index.

In [6]:
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(corpus)

print("Shape:", embeddings.shape)  # (50, 384)

d:\nihal\datacamp\ML practice\ml_env\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Shape: (50, 384)


## Step 4 — Create the index student-demo

An **index** in Pinecone is like a collection/table that holds vectors. We set:
- `dimension=384` → must match the embedding size.
- `metric="cosine"` → Pinecone ranks by cosine similarity (direction), where **higher score = more similar**. This is the opposite of FAISS `IndexFlatL2`, where **lower distance = more similar**.
- `ServerlessSpec` → the free serverless tier (no servers to manage).

We check first so re-running the cell does not error if the index already exists.

In [7]:
index_name = "student-demo"

existing = [idx.name for idx in pc.list_indexes()]
if index_name not in existing:
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    print("Created index:", index_name)
else:
    print("Index already exists:", index_name)

index = pc.Index(index_name)

Created index: student-demo


## Step 5 — Insert (upsert) the 50 documents

Pinecone calls inserting **upsert** (update + insert). Each vector needs:
- an `id` (string),
- the `values` (the 384 numbers — converted to a plain list),
- optional `metadata`. We store the original `text` inside the metadata so the query can show it back.

In [8]:
vectors = []
for i, (doc, emb, meta) in enumerate(zip(corpus, embeddings, metadatas)):
    vectors.append({
        "id": f"doc{i}",
        "values": emb.tolist(),
        "metadata": {**meta, "text": doc},
    })

index.upsert(vectors=vectors)

# upserts are processed asynchronously; wait a moment then check the count
time.sleep(5)
print(index.describe_index_stats())

DescribeIndexStatsResponse(dimension=384, total_vector_count=50, metric='cosine', namespaces=1)


## Step 6 — Query for semantic matches

We embed the query the **same way** we embedded the documents, then send the vector to Pinecone. We use the **same query as the FAISS task** so we can compare the top-3 results directly.

In [9]:
def search(query, top_k=3, flt=None):
    q_emb = model.encode([query])[0].tolist()
    res = index.query(
        vector=q_emb,
        top_k=top_k,
        include_metadata=True,
        filter=flt,
    )
    print("Query:", query)
    if flt:
        print("Filter:", flt)
    print()
    for rank, match in enumerate(res["matches"], start=1):
        print(f"Rank {rank} | Score: {match['score']:.4f}")
        print(f"Doc: {match['metadata']['text']}")
        print(f"Metadata: source={match['metadata']['source']}, year={match['metadata']['year']}")
        print()
    return res

# Same query used in the FAISS task
res_main = search("How does AI help with water pollution?", top_k=3)

Query: How does AI help with water pollution?

Rank 1 | Score: 0.6393
Doc: AI optimizes water distribution in drought-affected regions.
Metadata: source=scientific, year=2021

Rank 2 | Score: 0.5713
Doc: AI predicts air quality index for cities to issue health warnings.
Metadata: source=news, year=2022

Rank 3 | Score: 0.6456
Doc: AI tracks plastic pollution flow in rivers using image recognition.
Metadata: source=news, year=2020



Two more queries, to show semantic retrieval works across different topics:

In [10]:
_ = search("Which AI models help with climate and energy?", top_k=3)

Query: Which AI models help with climate and energy?

Rank 1 | Score: 0.7267
Doc: AI models simulate the impact of carbon emissions on global temperatures.
Metadata: source=news, year=2018

Rank 2 | Score: 0.6421
Doc: AI models predict drought conditions weeks in advance using climate models.
Metadata: source=research, year=2021

Rank 3 | Score: 0.7188
Doc: AI models analyze weather patterns to improve renewable energy predictions.
Metadata: source=news, year=2021



In [11]:
_ = search("How does AI monitor oceans and forests?", top_k=3)

Query: How does AI monitor oceans and forests?

Rank 1 | Score: 0.5902
Doc: AI monitors deforestation rates by comparing historical and current forest cover.
Metadata: source=news, year=2023

Rank 2 | Score: 0.5890
Doc: AI analyzes deforestation drivers using economic and satellite data together.
Metadata: source=research, year=2023

Rank 3 | Score: 0.5631
Doc: AI monitors endangered whale movements using underwater acoustic data.
Metadata: source=research, year=2024



## Step 7 — Bonus: metadata filtering

Because we stored `source` and `year` in the metadata, Pinecone can filter **before** ranking. Here we ask the same question but restrict it to `source = "scientific"` documents only — the same idea as the Chroma `where={...}` filter from Assignment 2.

In [12]:
_ = search(
    "How does AI help with water pollution?",
    top_k=3,
    flt={"source": "scientific"},
)

Query: How does AI help with water pollution?
Filter: {'source': 'scientific'}

Rank 1 | Score: 0.6393
Doc: AI optimizes water distribution in drought-affected regions.
Metadata: source=scientific, year=2021

Rank 2 | Score: 0.5328
Doc: AI can predict flood risks by analyzing satellite imagery and weather patterns.
Metadata: source=scientific, year=2018

Rank 3 | Score: 0.4281
Doc: AI recommends optimal routes for electric buses to lower emissions.
Metadata: source=scientific, year=2022



## Step 8 — Compare latency: Pinecone (cloud) vs FAISS (local)

FAISS searches in-memory on this laptop; Pinecone searches over the network. We time an average Pinecone query below. (In the FAISS task the same search is effectively instant — sub-millisecond — because there is no network hop.)

In [13]:
q_emb = model.encode(["How does AI help with water pollution?"])[0].tolist()

times = []
for _ in range(5):
    t0 = time.time()
    index.query(vector=q_emb, top_k=3, include_metadata=True)
    times.append((time.time() - t0) * 1000)  # milliseconds

print(f"Pinecone average query latency: {np.mean(times):.1f} ms over 5 runs")
print("(FAISS local search on 50 vectors is typically < 1 ms — no network involved.)")

Pinecone average query latency: 159.6 ms over 5 runs
(FAISS local search on 50 vectors is typically < 1 ms — no network involved.)


## Comparison table — FAISS vs Pinecone

| Aspect | FAISS (local) | Pinecone (cloud) |
|---|---|---|
| **Where it runs** | In memory on your laptop | Managed servers in the cloud |
| **Setup** | `pip install faiss-cpu`, no account | Sign up, get API key, create index |
| **Query speed** | Sub-millisecond (no network) | Tens to hundreds of ms (network hop) |
| **Similarity score** | L2 distance — *lower = closer* | Cosine — *higher = more similar* |
| **Metadata filtering** | Not built in (manual) | Built in (`filter=...`) |
| **Persistence** | Gone when kernel restarts (unless saved) | Stored permanently in the cloud |
| **Scaling** | Limited by your RAM | Scales to billions of vectors automatically |
| **Best for** | Prototyping, offline, small/medium data | Production apps, large data, multiple users |


## Reflection

**How does latency compare to FAISS (local)?**

FAISS is much faster *per query* because everything happens in local memory — no network. Pinecone adds tens to hundreds of milliseconds per query because each request travels to the cloud and back. So for tiny offline datasets FAISS wins on raw speed; the latency cell above shows the real difference on this machine.

**What advantages does a managed vector DB provide?**

- **No infrastructure to manage** — Pinecone handles servers, indexing, and scaling.
- **Persistence** — data survives restarts; FAISS lives in RAM and is lost unless you save it.
- **Scale** — handles millions/billions of vectors that would not fit in a laptop's RAM.
- **Built-in metadata filtering** and **concurrent access** for many users/apps at once.
- **Reliability** — backups, availability, and monitoring come for free.

**Top-3 comparison (FAISS vs Pinecone) for "How does AI help with water pollution?"**

Both return semantically related documents (water quality, irrigation, flood/pollution) rather than exact keyword matches — confirming that both do *semantic* search. The exact ranking differs for two reasons: (1) Pinecone here indexes **50** documents vs FAISS's **20**, and (2) Pinecone ranks by **cosine similarity** (higher = better) while our FAISS index used **L2 distance** (lower = better). Fill in the actual Pinecone top-3 from the Step 6 output next to your FAISS top-3 to complete the side-by-side.